In [1]:
!pip install mlflow


[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import mlflow
import mlflow.sklearn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

In [5]:
TRACKING_URI = "http://127.0.0.1:6969" # Ensure MLflow server is running at this address
EXPERIMENT_NAME = "Iris_Comparison"

# mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", EXPERIMENT_NAME)

MlflowException: API request to http://127.0.0.1:6969/api/2.0/mlflow/experiments/get-by-name failed with exception HTTPConnectionPool(host='127.0.0.1', port=6969): Max retries exceeded with url: /api/2.0/mlflow/experiments/get-by-name?experiment_name=Iris_Comparison (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x00000220D58556A0>: Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it'))

In [ ]:


def train_and_log(model_name, model, test_size=0.2, random_state=42):
    """Train and log a model with metrics to MLflow."""
    iris = load_iris()
    X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=test_size, random_state=random_state, stratify=iris.target
    )

    with mlflow.start_run(run_name=model_name):
        mlflow.log_param("model_type", model_name)
        mlflow.log_param("test_size", test_size)
        mlflow.log_param("random_state", random_state)

        # Fit model
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        # Compute metrics
        acc = accuracy_score(y_test, preds)
        prec = precision_score(y_test, preds, average="macro", zero_division=0)
        rec = recall_score(y_test, preds, average="macro", zero_division=0)
        f1 = f1_score(y_test, preds, average="macro", zero_division=0)

        # Log metrics
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("precision_macro", prec)
        mlflow.log_metric("recall_macro", rec)
        mlflow.log_metric("f1_macro", f1)

        # Log model artifact
        mlflow.sklearn.log_model(model, artifact_path="model")

        print(f"{model_name} logged successfully with accuracy={acc:.4f}")

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
train_and_log("RandomForestClassifier", rf_model)

In [ ]:
lr_model = LogisticRegression(max_iter=300, random_state=42)
train_and_log("LogisticRegression", lr_model)